# Brainstorming and Focus Group Quantitative Experimentation 1: **General US population** under **action correction** + **divergence intervention**

Can we use TinyTroupe to brainstorm product ideas?

In [1]:
import sys

from pprint import pprint

import tinytroupe
from tinytroupe.agent import TinyPerson
from tinytroupe.environment import TinyWorld
from tinytroupe.experimentation import InPlaceExperimentRunner
from tinytroupe.steering import Intervention
from tinytroupe.examples import *
from tinytroupe.validation import propositions
from tinytroupe.extraction import ResultsExtractor
from tinytroupe.utils.parallel import parallel_map_dict, parallel_map_cross
from tinytroupe.validation import hard_persona_adherence, persona_adherence, self_consistency, fluency, task_completion, divergence

# specific utilities
from common_utils import *


!!!!
DISCLAIMER: TinyTroupe relies on Artificial Intelligence (AI) models to generate content. 
The AI models are not perfect and may produce inappropriate or inaccurate results. 
For any serious or consequential use, please review the generated content before using it.
!!!!

Looking for default config on: C:\Users\pdasilva\repos\TinyTroupe and personal repos\TinyTroupe\tinytroupe\utils\..\config.ini
Found custom config on: c:\Users\pdasilva\repos\TinyTroupe and personal repos\TinyTroupe\publications\paper_artifacts_april-2026\config.ini
TinyTroupe version: 0.8.0
Current date and time (local): 2026-04-29 19:44:17
Current date and time (UTC):   2026-04-29 22:44:17

Current TinyTroupe configuration 
[OpenAI]
api_type = azure
azure_api_version = 2024-12-01-preview
model = gpt-5-mini
reasoning_model = o3-mini
vision_detail = auto
embedding_model = text-embedding-3-small
azure_embedding_model_api_version = 2023-05-15
max_completion_tokens = 128000
timeout = 300
max_attempts = 5
waiting_tim

## Parameters

In [2]:
full_mode = True  # set to True to run the full mode with all agents and tasks

# avoid displaying the communication, to make the output cleaner for eval
TinyPerson.communication_display = False

In [3]:
if full_mode:
    repetitions_per_task = 2
    simulation_steps = 5
    qty_agents = 12
    qty_proposals = 4

else:
    repetitions_per_task = 2
    simulation_steps = 5
    qty_agents = 4
    qty_proposals = 1


## Experiment setup

In [4]:
experiment_runner = InPlaceExperimentRunner("./brainstorming_and_focus_group_quantitative_experimentation_1.json")

experiment_runner.add_experiment("Control")
experiment_runner.add_experiment("Treatment")

2026-04-29 19:46:19,193 - MainThread(24040) - tinytroupe - WARNING - Configuration file './brainstorming_and_focus_group_quantitative_experimentation_1.json' exists and was loaded successfully. If you are trying to fully rerun the experiments, delete it first.
2026-04-29 19:46:19,203 - MainThread(24040) - tinytroupe - INFO - Experiment 'Control' already exists, nothihg to add.
2026-04-29 19:46:19,203 - MainThread(24040) - tinytroupe - INFO - Experiment 'Control' already exists, nothihg to add.
2026-04-29 19:46:19,206 - MainThread(24040) - tinytroupe - INFO - Experiment 'Treatment' already exists, nothihg to add.


In [5]:
experiment_runner.activate_next_experiment()

#experiment_runner.fix_active_experiment("Control")
#experiment_runner.fix_active_experiment("Treatment")

In [6]:
print(f"Running experiment {experiment_runner.get_active_experiment()}")

Running experiment Treatment


## Agents and populations

In [7]:

people = []
if not experiment_runner.has_finished_all_experiments():
    # load agents
    people = TinyPerson.load_specifications_from_folder("./population/usa_general_2")

    # filter to make it go faster?
    if qty_agents is not None:
        people = people[:qty_agents]

    # customize and print minibios 
    for person in people:
        ### person.import_fragment("./fragments/picky_customer.agent.fragment.json")
        print(person.minibio(extended=False))


Ariana Martinez-Brown is a 34 year old Kitchen Lead / Line Cook, American, currently living in Atlanta, Georgia (urban neighborhood, near a mixed commercial-residential corridor).
Ava Sinclair is a 21 year old Junior Software Developer (Front-end) / Freelance Game Designer and Graphic Designer, American, currently living in Chicago, Illinois (urban, near Bronzeville neighborhood).
Benjamin Hartley is a 42 year old Senior Engineering Manager (Director-level responsibilities), American, currently living in San Francisco, California, USA.
Carmen Alvarez-Johnson is a 59 year old Community Health Outreach Coordinator / Part-time Licensed Practical Nurse (LPN), American, currently living in Atlanta, Georgia, USA (urban neighborhood, Eastside near a community health clinic and public transit).
Carolyn Whitman is a 61 year old Senior Project Manager — Civil Infrastructure, American, currently living in Madison, Wisconsin (mid-sized Midwestern city).
Charlotte Mercer is a 14 year old Student; J

In [8]:
len(people)

12

In [9]:
# divide people in several groups of 5
people_groups = []
for i in range(0, len(people), 5):
    people_groups.append(people[i:i+5]
    )

len(people_groups)

3

In [10]:
# The experiment refers to customers

if experiment_runner.get_active_experiment() == "Control":
    for person in people:
        person.action_generator.enable_reasoning_step = False
        person.action_generator.enable_quality_checks = False

elif experiment_runner.get_active_experiment() == "Treatment":    
    for person in people:
       person.action_generator.enable_reasoning_step = False
       person.action_generator.enable_quality_checks = True
       person.action_generator.max_attempts = 5
       person.action_generator.enable_regeneration = True
       person.action_generator.quality_threshold = 5

## Proposals

In [11]:
proposals = [
    {"theme": "Food and Nutrition (food itself, consumption, preparation, transportation, storage)", 
    "objective": "ideas for new food products, either new foods, food services, food experiences, "+\
                "or food preparation tools." },
    {"theme": "Travel and Tourism (travel, tourism, hospitality, leisure)",
    "objective": "ideas for new travel and tourism services, experiences, or products." },
    {"theme": "Health and Wellbeing (health, wellness, fitness, beauty)",
    "objective": "ideas for new health and wellbeing services, experiences, or products." },
    {"theme": "Economics and Finance (economics, finance, business, work)",
    "objective": "ideas for new economic and financial services, experiences, or products." },
    {"theme": "Technology and Innovation (technology, innovation, science, research)",
    "objective": "ideas for new technology and innovation services, experiences, or products." }
]

if not full_mode:
    proposals = proposals[:qty_proposals]

## Auxiliary functions

In [12]:
def brainstorming_battery(agents, proposals, interventions, agent_propositions, environment_propositions, 
                          repetitions = 5, simulation_steps=10): 
    
    agent_propositions_scores = {}
    environment_propositions_scores = {}

    experiments_count = 0
    total_expected_experiments = len(proposals) * repetitions #* len(agents)

    # TODO remove?
    #
    # Add intervention to prevent agents from being too quiet.
    #for agent in agents:
    #    intervention = \
    #        Intervention(agent)\
    #            .set_propositional_precondition(propositions.quiet_recently)\
    #            .set_effect(lambda target: target.think("""
    #                                                    I will say something now, I've been too quiet for a while. If I am uncomfortable, 
    #                                                    or can't think of a proper response,
    #                                                    I can always say something like "I don't want to talk about this",
    #                                                    or propose another topic.
    #                                                    """))
    #    interventions.append(intervention)

    # loop over proposals and repetitions
    for proposal in proposals:

        objective = proposal["objective"]
        theme = proposal["theme"]

        for i in range(repetitions):
            print("\n############## STARTING A NEW RESEARCH SESSION #################")
            print(f"Overall experiment number: {experiments_count+1} / {total_expected_experiments}")
            print(f"Discussion objective: {objective}")
            print(f"Trial number: {i+1}")
            print(f"Agents: {agents}")

            # clear the episodic memory of all agents
            for person in agents:
                person.clear_episodic_memory()

            world = TinyWorld(agents=agents, interventions=interventions)
            
            # Participants introduce themselves
            world.broadcast(f"""
                Hello everyone! Let's start by introducing ourselves, and mentioning problems we face in our daily personal
                and professional lives related to the following theme: {theme}
                
                Please:
                  - present yourself and your background;
                  - present some key personal problems related to the theme;
                  - present some key problems related to the theme that you face in your work;
                  - present some key problems related to the theme that you see in your industry as a whole.
                  
                Don't discuss solutions yet, just the problems you face and see others facing.
                """)
            world.run(1)
            
            # now to the brainstorming session itself
            world.broadcast(f"""
                Folks, your mission is to brainstorm {objective}. 
                Please follow these guidelines:
                  - give a unique and informative name to each idea you propose, so that it is easy to refer to it. Say it like "Idea name: '<name of the idea>'".;
                  - explain why you think it is a good idea, and what problem it solves, and how you feel about it;
                  - your ideas should be new complete, self-contained, products or services, not features for other existing products or services;
                  - think of creative ideas that would somehow help you in both in your personal and professional lives.
                  - create as many different and unique ideas as you can during the brainstorming session. Each idea must be **completely** different from the others 
                    (either by yourself or by others), and not just a variation of an existing idea.
                    and not just a variation of an existing idea.
                  - you should criticize each other's ideas, in order to make sure they are as
                    good as possible, but no more than once per idea.
                  - you should also provide suggestions for improvement to each other's ideas, in order to make them as good as possible, 
                    but no more than once per idea.
                  - regardless of critique or complement, you **must** primarily propose new ideas quickly, 
                    not just build on existing ones. 
                  - propose one idea at a time, instead of proposing multiple ideas at once, to allow appropriate discussion.
                  - you should **not** propose ideas that are too similar to each other, or to the ones already proposed by others.
                  - before saying anything, THINK deeply about yourself, your beliefs, interests, needs, life, etc., to come up with ideas that are
                    truly unique and different from the ones already proposed by others.
                   
                Please start the discussion now.
                """)
            world.run(simulation_steps)

            # extract and count ideas
            rapporteur = agents[0]  # the first agent is the rapporteur
            rapporteur.listen_and_act("Can you please consolidate the ideas that the group came up with? Provide a lot of details on each idea, and complement anything missing.")
            ideas = ResultsExtractor().extract_results_from_agent(rapporteur, 
                                    extraction_objective="Consolidates the ideas that the group came up with, explaining each idea as an item of a list." \
                                                        "Add information about: what problem the idea solves; to which target audience it is meant." \
                                                        "how is it different from competing, existing, products.", 
                                    situation="A focus group to brainstorm new product ideas.",
                                    fields= ["name", "description", "problem", "target_audience", "competition_analysis"],
                                    fields_hints={"ideas": "must be the root of the resulting dictionary."},)
            pprint(ideas)
            if "ideas_qty" not in environment_propositions_scores:
                environment_propositions_scores["ideas_qty"] = []
            if ideas is not None and "ideas" in ideas and isinstance(ideas["ideas"], list):
                environment_propositions_scores["ideas_qty"].append(len(ideas["ideas"]))

            # Evaluate environment propositions in parallel
            env_results = parallel_map_dict(
                environment_propositions,
                lambda item: item[1].copy().score(
                    world, 
                    claim_variables={"task_description": f"A brainstorming or focus group session was run about: {objective}."}, 
                    return_full_response=True
                )
            )
            
            # Process environment results
            for k, result in env_results.items():
                if k not in environment_propositions_scores:
                    environment_propositions_scores[k] = []
                environment_propositions_scores[k].append(result["value"])
                print("value: ", result["value"])
                print("justification: ", result["justification"])
                print("reasoning: ", result["reasoning"])

            # Evaluate agent propositions across all agents in parallel
            agent_results = parallel_map_cross(
                [agents, agent_propositions.items()],
                lambda agent, prop_item: (
                    prop_item[0],  # proposition key
                    prop_item[1].copy().score(agent, return_full_response=True)  # result
                )
            )
            
            # Process agent results
            for k, result in agent_results:
                if k not in agent_propositions_scores:
                    agent_propositions_scores[k] = []
                if result is not None:
                    agent_propositions_scores[k].append(result["value"])
                    print("value: ", result["value"])
                    print("justification: ", result["justification"])
                    print("reasoning: ", result["reasoning"])
                    print("\n\n")
                else:
                    print(f"*****WARNING:***** Agent did not respond to proposition {k}.")
            #
            ##for k, proposition in agent_propositions.items():
            ##    for person in world.agents:
            ##        result = proposition.copy().score(person, return_full_response=True)
            ##        
            ##        if k not in agent_propositions_scores:
            ##            agent_propositions_scores[k] = []
            ##        agent_propositions_scores[k].append(result["value"])
            ##
            ##        print("value: ", result["value"])
            ##        print("justification: ", result["justification"])
            ##        print("reasoning: ", result["reasoning"])
            ##        print("\n\n")
            ##
            
            experiments_count += 1
            print("\n\n")

    return agent_propositions_scores, environment_propositions_scores



## Perform experiment

In [13]:
agent_propositions_scores={}
environment_propositions_scores={}

In [14]:
def brainstorm(people):
    global agent_propositions_scores, environment_propositions_scores
    if not experiment_runner.has_finished_all_experiments():

        interventions = []
        if experiment_runner.get_active_experiment() == "Treatment":
            interventions = \
                Intervention.create_for_each(people)\
                    .set_functional_precondition(lambda target: target.actions_count >=7)\
                    .set_textual_precondition(
                        """
                        AGENT IS NOT PROPOSING COMPLETELY NEW PRODUCT/SERVICE IDEAS ANYMORE:
                        The last **entirely** new product/service idea proposed by this agent, if any, was proposed by him/her **more** than 10 of simulation events ago.
                        That is to say, the agent has not proposed any new product/service idea in the last 10 of his/her simulation trajectory events.
                        Additional features, variations of or other refinements to product/service ideas already proposed are NOT considered new!

                        How to compute the steps gap:
                        1. Determine the current next event number (N); and the last event number in which the agent proposed a new product/service idea (M).
                            This information can be found in the simulation trajectory.
                        2. Compute the **difference** beteween the current next event number and the last event number in which the agent proposed a new product/service idea: D = N - M
                        3. The proposition is true if, and only if, the difference D is **greater than** 10.
                        """)\
                    .set_effect(lambda target: target.think("""
                                                            I need to propose additional, **completelly** new and different, product/service ideas. This was part of the requirement for this session.
                                                            I will propose an entirely **new** idea now, I **cannot** repeat or refine previous ideas! I cannot make variations
                                                            of previous ideas (e.g., "XYZ for A", "XYZ for B", "XYZ for Z" are repetitive, there should be only one "XYZ"), 
                                                            I need to think of something **entirely** new and different.
                                                            To help me avoid repeating previous ideas, I'll now explicitly THINK about all the ideas already given by myself or
                                                            others, and then, based on that, I'll think again about a new unique idea.
                                                            """))

                                                            
        tmp_agent_propositions_scores, tmp_environment_propositions_scores = \
            brainstorming_battery(
                agents=people,
                proposals=proposals,
                interventions=interventions,    
                agent_propositions={
                    "Hard Persona Adherence": hard_persona_adherence,
                    "Self-consistency": self_consistency,
                    "Fluency": fluency
                },
                environment_propositions={
                    "Task Completion": task_completion,
                    "Divergence": divergence
                },
                repetitions=repetitions_per_task,
                simulation_steps=simulation_steps
            )

        pprint("NEW AGENT PROPOSITIONS SCORES")
        pprint(tmp_agent_propositions_scores)
        print("\n\n")
        pprint("NEW ENVIRONMENT PROPOSITIONS SCORES")
        pprint(tmp_environment_propositions_scores)

        # merge the scores lists
        agent_propositions_scores = merge_dicts_of_lists(tmp_agent_propositions_scores, agent_propositions_scores)
        environment_propositions_scores = merge_dicts_of_lists(tmp_environment_propositions_scores, environment_propositions_scores)

        return agent_propositions_scores, environment_propositions_scores

In [15]:
brainstorm(people_groups[0]) if len(people_groups) > 0 else None


############## STARTING A NEW RESEARCH SESSION #################
Overall experiment number: 1 / 10
Discussion objective: ideas for new food products, either new foods, food services, food experiences, or food preparation tools.
Trial number: 1
Agents: [TinyPerson(name='Ariana Martinez-Brown'), TinyPerson(name='Ava Sinclair'), TinyPerson(name='Benjamin Hartley'), TinyPerson(name='Carmen Alvarez-Johnson'), TinyPerson(name='Carolyn Whitman')]
2026-04-29 19:46:24,674 - MainThread(24040) - tinytroupe - INFO - [TinyWorld 1] Running world simulation step 1 of 1.


───────────────────────────────────────────── TinyWorld 1 step 1 of 1 ─────────────────────────────────────────────

2026-04-29 19:46:24,719 - MainThread(24040) - tinytroupe - INFO - [TinyWorld 1] No timedelta provided, so the datetime was not advanced.


2026-04-29 19:46:28,619 - MainThread(24040) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-29 19:46:29,702 - MainThread(24040) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-29 19:47:08,241 - MainThread(24040) - httpx - INFO - HTTP Request: POST https://azureai-prototyping-ai.openai.azure.com/openai/deployments/gpt-5-mini/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"
2026-04-29 19:47:10,705 - MainThread(24040) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-29 19:47:10,735 - MainThread(24040) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-29 19:47:54,879 - MainThread(24040) - httpx - INFO - HTTP Request: POST https://azureai-prototyping-ai.openai.azure.com/openai/deployments/gpt-5-mini/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"
2026-04-29 19:47:56,682 - MainThread(24040) - tinytroupe - INFO 

───────────────────────────────────────────── TinyWorld 1 step 1 of 5 ─────────────────────────────────────────────

...

──────────────────────────────────────────── TinyWorld 10 step 5 of 5 ─────────────────────────────────────────────

2026-04-30 03:13:47,062 - MainThread(24040) - tinytroupe - INFO - [TinyWorld 10] No timedelta provided, so the datetime was not advanced.
2026-04-30 03:13:48,080 - MainThread(24040) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-30 03:13:48,100 - MainThread(24040) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-30 03:14:32,068 - MainThread(24040) - httpx - INFO - HTTP Request: POST https://azureai-prototyping-ai.openai.azure.com/openai/deployments/gpt-5-mini/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"
2026-04-30 03:14:36,574 - MainThread(24040) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-30 03:14:36,667 - MainThread(24040) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-30 03:15:18,749 - MainThread(24040) - httpx - INFO - HTTP Request: POST https://azureai-prototyping-ai.openai.azure.com/openai/deployments/gpt

 ⚡  <<Intervention 5>> Triggered, effects are being applied... 
          > The context gives N = 51 ("The last agent simulation trajectory event number was 50,
          > thus the current number of the NEXT POTENTIAL TRAJECTORY EVENT is 51"). The last event
          > where Carolyn explicitly proposed an entirely new product/service is event #39: she
          > 'TALK' introduced "PermitBuddy Kit" (full brief description and core components).
          > Earlier new ideas she proposed include #3/#14 (Neighborhood Resilience Hub) and #24
          > (PlanMatch Field Kit), but these precede #39. After event #39, her actions are not new
          > idea proposals: #40 is DONE, #41–#43 are others' comments, #44 THINK about Carmen's
          > idea, #45 TALK providing pragmatic suggestions for Carmen's "Mano Amiga" (not a new idea
          > by Carolyn), #46 DONE, and subsequent events up to #50 are others' contributions. No
          > later event shows Carolyn presenting another distinct, entirely new product/service.
          > Thus M = 39 and D = 51 - 39 = 12, which is greater than 10, so the proposition "agent is
          > not proposing completely new product/service ideas anymore" is true under the stated
          > rule. (confidence = 1.0)  Functional precondition was met.

2026-04-30 03:17:19,041 - ThreadPoolExecutor-77_2(31148) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-30 03:17:19,059 - ThreadPoolExecutor-77_4(9712) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-30 03:17:19,063 - ThreadPoolExecutor-77_3(14764) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-30 03:17:19,070 - ThreadPoolExecutor-77_1(27864) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-30 03:17:19,085 - ThreadPoolExecutor-77_0(36552) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-30 03:17:19,126 - ThreadPoolExecutor-77_2(31148) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-30 03:17:19,150 - ThreadPoolExecutor-77_4(9712) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-30 03:17:19,175 - ThreadPoolExecutor-77_3(14764) - tinytroupe - INFO - Waiting 5.0 secon

({'Hard Persona Adherence': [9,
   9,
   9,
   9,
   9,
   9,
   7,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   6,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   7,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9],
  'Self-consistency': [9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   7,
   9,
   9,
   9,
   9,
   9,
   9,
   2,
   9,
   9,
   9,
   9,
   8,
   9,
   9,
   9,
   9,
   9,
   9,
   8,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   3,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9],
  'Fluency': [8,
   9,
   3,
   8,
   8,
   7,
   7,
   9,
   8,
   7,
   8,
   8,
   5,
   7,
   3,
   8,
   8,
   8,
   8,
   6,
   7,
   8,
   7,
   7,
   7,
   8,
   8,
   8,
   6,
   1,
   7,
   8,
   8,
   7,
   7,
   8,
   8,
   8,
   6,
   7,
   8,
   7,
   7,
   6,
   7,
   7,
   7,
   8,
   7,
   6]},
 {'ideas_qty': [11, 13, 11, 15, 13, 12,

In [16]:
brainstorm(people_groups[1]) if len(people_groups) > 1 else None


############## STARTING A NEW RESEARCH SESSION #################
Overall experiment number: 1 / 10
Discussion objective: ideas for new food products, either new foods, food services, food experiences, or food preparation tools.
Trial number: 1
Agents: [TinyPerson(name='Charlotte Mercer'), TinyPerson(name='Elliot James Prescott'), TinyPerson(name='Eloise Gardner'), TinyPerson(name='Ethan Nakamura'), TinyPerson(name='Evelyn Bradford')]
2026-04-30 03:29:27,781 - MainThread(24040) - tinytroupe - INFO - [TinyWorld 11] Running world simulation step 1 of 1.


──────────────────────────────────────────── TinyWorld 11 step 1 of 1 ─────────────────────────────────────────────

2026-04-30 03:29:27,789 - MainThread(24040) - tinytroupe - INFO - [TinyWorld 11] No timedelta provided, so the datetime was not advanced.


2026-04-30 03:29:28,876 - MainThread(24040) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-30 03:29:28,884 - MainThread(24040) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-30 03:30:02,853 - MainThread(24040) - httpx - INFO - HTTP Request: POST https://azureai-prototyping-ai.openai.azure.com/openai/deployments/gpt-5-mini/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"
2026-04-30 03:30:03,893 - MainThread(24040) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-30 03:30:03,900 - MainThread(24040) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-30 03:30:46,047 - MainThread(24040) - httpx - INFO - HTTP Request: POST https://azureai-prototyping-ai.openai.azure.com/openai/deployments/gpt-5-mini/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"
2026-04-30 03:30:46,728 - MainThread(24040) - tinytroupe - INFO 

──────────────────────────────────────────── TinyWorld 11 step 1 of 5 ─────────────────────────────────────────────

...

 ⚡  <<Intervention 9>> Triggered, effects are being applied... 
          > Concrete evidence and precise references from the trajectory that support the result: -
          > The context explicitly states the next potential trajectory event number is 48 (so N =
          > 48). - Ethan Nakamura proposed the idea "CallAnchor" at event #23 (bold green3 [TALK]);
          > this is an explicit new product/service idea. - Ethan Nakamura later proposed another
          > explicit new product/service idea, "Fixerhood", at event #37 (bold green3 [TALK]) and
          > then marked DONE at #38. - After event #37 there are many events (responses from other
          > participants, Ethan THINK/DONE markers, feedback, etc.), but no further event in which
          > Ethan proposes an entirely new product or service idea. For example: events #38–#47 are
          > other participants' idea posts, Ethan's thoughts, or feedback, not new distinct idea
          > proposals. - Following the rule for D: D = N - M = 48 - 37 = 11. Since the proposition
          > requires D > 10 to be true, and 11 > 10, the proposition holds. - Additional note: the
          > proposition excludes refinements or variations of existing ideas; both "CallAnchor" and
          > "Fixerhood" are distinct, self-contained ideas, and the last distinct one was at #37.
          > Therefore the agent has not proposed any entirely new product/service idea within the
          > last 10 of his/her simulation trajectory events (the gap is 11 events). (confidence =
          > 1.0)  Functional precondition was met.

2026-04-30 21:03:43,678 - MainThread(24040) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-30 21:03:43,766 - MainThread(24040) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-30 21:04:40,379 - MainThread(24040) - httpx - INFO - HTTP Request: POST https://azureai-prototyping-ai.openai.azure.com/openai/deployments/gpt-5-mini/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"


 ⚡  <<Intervention 10>> Triggered, effects are being applied... 
          > The proposition asks whether the agent has not proposed any entirely new product/service
          > ideas in the last 10 of her simulation trajectory events, i.e., whether the gap D =
          > (next event number) - (last event number with an entirely new idea) is greater than 10.
          > Concrete evidence from the trajectory: - N is explicitly given as 51 in the context:
          > "The last agent simulation trajectory event number was 50, thus the current number of
          > the NEXT POTENTIAL TRAJECTORY EVENT is 51." (so N = 51). - Evelyn's last explicit,
          > entirely new product/service idea is at event #38: she speaks the idea "SkillBurst"
          > (event #38: bold green3 TALK: Idea name: "SkillBurst" ...). Prior new idea at #23 was
          > "DataPassport", but the later new idea at #38 supersedes it as the most recent new idea.
          > - After event #38 Evelyn's subsequent entries are DONE at #39 and then later she
          > provides feedback and discussion (#49, #45 etc.), but she does not propose another
          > entirely new, distinct product/service idea in any event between #39 and #50. Many
          > events in that range are other participants proposing ideas or Evelyn responding, but
          > none are a new product/service idea from Evelyn herself.  Therefore M = 38, so D = 51 -
          > 38 = 13. Because 13 > 10, the condition in the proposition is satisfied: Evelyn has not
          > proposed a completely new product/service idea in the last 10 of her trajectory events.
          > This matches the rule that refinements, feedback, or comments are not counted as new
          > ideas; only event #38 qualifies as her last entirely new idea.  Thus the proposition is
          > True. (confidence = 1.0)  Functional precondition was met.

2026-04-30 21:04:44,538 - ThreadPoolExecutor-157_0(34172) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-30 21:04:44,632 - ThreadPoolExecutor-157_4(52564) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-30 21:04:44,748 - ThreadPoolExecutor-157_0(34172) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-30 21:04:44,858 - ThreadPoolExecutor-157_4(52564) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-30 21:04:44,906 - ThreadPoolExecutor-157_3(47564) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-30 21:04:45,113 - ThreadPoolExecutor-157_3(47564) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-30 21:04:46,698 - ThreadPoolExecutor-157_2(31648) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-30 21:04:47,099 - ThreadPoolExecutor-157_2(31648) - t

({'Hard Persona Adherence': [9,
   9,
   9,
   9,
   9,
   9,
   7,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   6,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   7,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   7,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   7,
   9,
   9,
   6,
   9,
   9,
   9,
   9,
   9,
   9,
   5,
   9,
   9,
   9,
   9,
   7,
   9,
   7,
   7,
   9,
   9,
   9,
   9,
   7,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   7,
   9],
  'Self-consistency': [9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   7,
   9,
   9,
   9,
   9,
   9,
   9,
   2,
   9,
   9,
   9,
   9,
   8,
   9,
   9,
   9,
   9,
   9,
   9,
   8,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   3,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   3,
   9,

In [17]:
brainstorm(people_groups[2]) if len(people_groups) > 2 else None


############## STARTING A NEW RESEARCH SESSION #################
Overall experiment number: 1 / 10
Discussion objective: ideas for new food products, either new foods, food services, food experiences, or food preparation tools.
Trial number: 1
Agents: [TinyPerson(name='Gloria Rosario'), TinyPerson(name='Harper Sullivan')]
2026-04-30 21:20:11,770 - MainThread(24040) - tinytroupe - INFO - [TinyWorld 21] Running world simulation step 1 of 1.


──────────────────────────────────────────── TinyWorld 21 step 1 of 1 ─────────────────────────────────────────────

2026-04-30 21:20:11,780 - MainThread(24040) - tinytroupe - INFO - [TinyWorld 21] No timedelta provided, so the datetime was not advanced.
2026-04-30 21:20:13,744 - MainThread(24040) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-30 21:20:13,764 - MainThread(24040) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-30 21:21:16,225 - MainThread(24040) - httpx - INFO - HTTP Request: POST https://azureai-prototyping-ai.openai.azure.com/openai/deployments/gpt-5-mini/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"
2026-04-30 21:21:17,117 - MainThread(24040) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-30 21:21:17,125 - MainThread(24040) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-30 21:22:04,729 - MainThread(24040) - httpx - INFO - HTTP Request: POST https://azureai-prototyping-ai.openai.azure.com/openai/deployments/gpt

──────────────────────────────────────────── TinyWorld 21 step 1 of 5 ─────────────────────────────────────────────

2026-04-30 21:25:34,134 - MainThread(24040) - tinytroupe - INFO - [TinyWorld 21] No timedelta provided, so the datetime was not advanced.
2026-04-30 21:25:35,200 - MainThread(24040) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-30 21:25:35,218 - MainThread(24040) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-30 21:26:26,802 - MainThread(24040) - httpx - INFO - HTTP Request: POST https://azureai-prototyping-ai.openai.azure.com/openai/deployments/gpt-5-mini/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"
2026-04-30 21:26:28,423 - MainThread(24040) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-30 21:26:28,436 - MainThread(24040) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-30 21:27:27,237 - MainThread(24040) - httpx - INFO - HTTP Request: POST https://azureai-prototyping-ai.openai.azure.com/openai/deployments/gpt

...

 ⚡  <<Intervention 11>> Triggered, effects are being applied... 
          > The context shows the next event number is N = 34 (explicitly stated at the end of the
          > trajectory). Gloria's new idea proposals appear at event #7 ('Manos y Móviles') and at
          > event #23 ('La Cocina Saludable del Barrio'); no later events show her proposing another
          > entirely new product/service idea. Events after #23 are 'DONE' markers or responses to
          > Harper's suggestions (practical tweaks, pilot planning, volunteer recruitment), which
          > are clarifications, implementation details, or refinements rather than new, distinct
          > product/service ideas. Thus M = 23. Computing D = 34 - 23 = 11, which is greater than
          > 10, satisfies the criterion in the proposition. Therefore the claim that the agent has
          > not proposed any completely new product/service idea in the last 10 of her trajectory
          > events is true. (confidence = 1.0)  Functional precondition was met.

2026-05-01 03:27:23,405 - MainThread(24040) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-01 03:27:23,423 - MainThread(24040) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-01 03:28:29,274 - MainThread(24040) - httpx - INFO - HTTP Request: POST https://azureai-prototyping-ai.openai.azure.com/openai/deployments/gpt-5-mini/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"


 ⚡  <<Intervention 12>> Triggered, effects are being applied... 
          > The proposition asks whether the agent has not proposed any entirely new product/service
          > idea in the last 10 of his/her simulation trajectory events, i.e., whether D = N - M >
          > 10. From the provided trajectory: the next event number N = 36. Harper's distinct,
          > entirely new idea proposals in the trajectory are at events #3 (TrailCare Connect), #9
          > (GearShare Health Hub), and #25 (Neighborhood Health Scouts). Although TrailCare Connect
          > and GearShare appear again later (events #14 and #20), those are repeats of earlier
          > ideas and thus do not count as new, per the proposition's rule that
          > refinements/variations/repeats are NOT considered new. After event #25, Harper's
          > activity consists of comments, coordination, and feedback (events #26–#35), but no new,
          > complete, distinct product/service idea proposals. Therefore M = 25, giving D = 36 - 25
          > = 11. Because 11 is greater than 10, the condition in the proposition is satisfied.
          > Hence the statement "AGENT IS NOT PROPOSING COMPLETELY NEW PRODUCT/SERVICE IDEAS
          > ANYMORE" is True with respect to the provided trajectory. I am confident in this
          > assessment because the timeline and event contents clearly identify the last unique idea
          > at event #25 and the explicit N = 36. (confidence = 1.0)  Functional precondition was
          > met.

2026-05-01 03:28:30,449 - ThreadPoolExecutor-237_0(52240) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-01 03:28:30,454 - ThreadPoolExecutor-237_1(42116) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-01 03:28:30,507 - ThreadPoolExecutor-237_0(52240) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-01 03:28:30,509 - ThreadPoolExecutor-237_1(42116) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-01 03:29:40,982 - ThreadPoolExecutor-237_1(42116) - httpx - INFO - HTTP Request: POST https://azureai-prototyping-ai.openai.azure.com/openai/deployments/gpt-5-mini/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"
2026-05-01 03:29:41,950 - ThreadPoolExecutor-237_1(42116) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-01 03:29:41,985 - ThreadPoolExecutor-237_1(42116) - tinytroupe - INFO - Waiting 5.0 sec

({'Hard Persona Adherence': [9,
   9,
   9,
   9,
   9,
   9,
   7,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   6,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   7,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   7,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   7,
   9,
   9,
   6,
   9,
   9,
   9,
   9,
   9,
   9,
   5,
   9,
   9,
   9,
   9,
   7,
   9,
   7,
   7,
   9,
   9,
   9,
   9,
   7,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   7,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9],
  'Self-consistency': [9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   7,
   9,
   9,
   9,
   9,
   9,
   9,
   2,
   9,
   9,
   9,
   9,
   8,
   9,
   9,
   9,
   9,
   9,
   9,
   8,
   9,
   9,
   9,
   9,
   9,

In [18]:
brainstorm(people_groups[3]) if len(people_groups) > 3 else None

In [19]:
brainstorm(people_groups[4]) if len(people_groups) > 4 else None

## Extract results and analyze

In [20]:
if experiment_runner.get_active_experiment() in ["Control", "Treatment"]:
    combined_scores = {**agent_propositions_scores, **environment_propositions_scores}
    experiment_runner.add_experiment_results(combined_scores, experiment_name=experiment_runner.get_active_experiment()) 
    
    plot_scores(combined_scores)

else:
    print("Experiment finished. No more experiments to run.")

{'Divergence': [8,
                9,
                6,
                9,
                9,
                9,
                9,
                8,
                8,
                7,
                9,
                9,
                9,
                9,
                9,
                9,
                8,
                9,
                8,
                8,
                3,
                7,
                6,
                7,
                9,
                8,
                7,
                8,
                9,
                9],
 'Fluency': [8,
             9,
             3,
             8,
             8,
             7,
             7,
             9,
             8,
             7,
             8,
             8,
             5,
             7,
             3,
             8,
             8,
             8,
             8,
             6,
             7,
             8,
             7,
             7,
             7,
             8,
             

,Proposition,Average Score,Standard Deviation,Count
0,Hard Persona Adherence,8.766667,0.730297,120.0
1,Self-consistency,8.766667,1.026811,120.0
2,Fluency,6.733333,1.809096,120.0
3,ideas_qty,10.107143,3.965960,28.0
4,Task Completion,9.000000,0.000000,30.0
5,Divergence,8.066667,1.337350,30.0


In [21]:
if experiment_runner.has_finished_all_experiments():
    print("All experiments have been finished.")
    print(f"STATISTICTS: Control vs")
    pprint(experiment_runner.run_statistical_tests(control_experiment_name='Control'))

    # plot scores of both experiments
    experiment_control_scores = experiment_runner.get_experiment_results("Control")
    experiment_treatment_scores = experiment_runner.get_experiment_results("Treatment")
    
    
    plot_scores(experiment_control_scores)
    plot_scores(experiment_treatment_scores)

else:
    print("Not all experiments have been finished. RESTART AND RERUN.")

Not all experiments have been finished. RESTART AND RERUN.


In [22]:
experiment_runner.finish_active_experiment()

2026-05-01 03:44:44,778 - MainThread(24040) - tinytroupe - INFO - Experiment 'Treatment' marked as finished.
2026-05-01 03:44:44,780 - MainThread(24040) - tinytroupe - INFO - All experiments have been finished.


True